# Handling Imbalanced Data (fraud / churn style)

In fraud detection, churn, rare-disease screening and click-through, the *interesting* class is a tiny minority — maybe 1 positive per 20 negatives. A model can score **95%+ accuracy by predicting "not fraud" for everyone** and still be worthless, because it never catches a single fraud.

This notebook builds a fraud-style dataset (95% negative / 5% positive) and walks through:

1. **Why accuracy misleads** — a trivial majority-class baseline vs. a real classifier.
2. **Three remedies**, each trained and evaluated:
   - `class_weight='balanced'` — tell the loss to care more about the rare class.
   - **SMOTE** — synthesize new minority examples so the *training* set is balanced.
   - **Threshold tuning** — stop using 0.5; pick the operating point from the precision-recall trade-off.
3. **Evaluation with precision-recall curves** and `average_precision_score`, plus confusion matrices — the right lens under heavy imbalance.

**Golden rule up front:** we only ever resample / rebalance the **training set**. The test set must keep the real-world 95/5 ratio, or our metrics become fiction.

### A note on `imbalanced-learn`

The standard library for this is **imbalanced-learn** (`imblearn`), which ships a battle-tested `SMOTE`:

```bash
pip install imbalanced-learn
```

It may not be installed in every environment, so below we **try to import it and fall back to a compact hand-rolled SMOTE** if it's missing. The notebook runs identically either way — everything else uses only numpy, pandas, scikit-learn, matplotlib, scipy and seaborn.

In [ ]:
import numpy as np                     # numeric arrays + the math for hand-rolled SMOTE
import pandas as pd                    # tabular display of the metric comparison table
import matplotlib.pyplot as plt        # plotting (PR curves, confusion matrices)
import seaborn as sns                  # nicer confusion-matrix heatmaps

from sklearn.datasets import make_classification          # synthetic imbalanced dataset
from sklearn.model_selection import train_test_split      # stratified train/test split
from sklearn.linear_model import LogisticRegression       # our classifier throughout
from sklearn.neighbors import NearestNeighbors            # needed for the hand-rolled SMOTE fallback
from sklearn.metrics import (
    accuracy_score,            # the metric we will show is MISLEADING here
    precision_score,           # of predicted-positives, how many were right
    recall_score,              # of actual-positives, how many we caught
    f1_score,                  # harmonic mean of precision & recall
    average_precision_score,   # area under the precision-recall curve (our headline metric)
    precision_recall_curve,    # the (precision, recall) trade-off across all thresholds
    confusion_matrix,          # TP / FP / FN / TN table at a chosen threshold
)

# One global seed so the dataset, split, model and hand-rolled SMOTE are all reproducible.
SEED = 42
np.random.seed(SEED)
rng = np.random.default_rng(SEED)   # modern NumPy Generator used inside the SMOTE fallback

sns.set_theme(style="whitegrid")    # consistent, clean look for every plot

## 1. Build a fraud-style imbalanced dataset

`make_classification` with `weights=[0.95, 0.05]` gives roughly **95% class-0 (legit) and 5% class-1 (fraud)** — the rare positive we care about.

We then take a **stratified** train/test split. `stratify=y` forces the *same* 95/5 ratio into both splits; a plain random split could, by chance, starve the test set of the handful of positives and make evaluation noisy.

In [ ]:
# n_samples large enough that 5% still leaves a few hundred positives to learn from.
X, y = make_classification(
    n_samples=10000,        # total rows
    n_features=20,          # 20 features...
    n_informative=6,        # ...6 actually carry signal
    n_redundant=2,          # 2 are linear combos of the informative ones
    n_clusters_per_class=2, # a bit of structure per class so it isn't trivially separable
    weights=[0.95, 0.05],   # <-- the imbalance: ~95% negative, ~5% positive (fraud)
    flip_y=0.01,            # 1% label noise -> realistic, no perfect separation
    class_sep=0.9,          # moderate separation: hard but learnable
    random_state=SEED,
)

# Stratified split keeps the 95/5 ratio identical in train and test.
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=SEED, stratify=y
)

# Report the class balance so the imbalance is concrete.
def class_ratio(name, labels):
    pos = int(labels.sum())                 # count of positives (fraud)
    tot = len(labels)
    print(f"{name:>6}: {tot:5d} rows | positives = {pos:4d} ({100*pos/tot:.2f}%)")

print("Class balance (positive = fraud):")
class_ratio("full",  y)
class_ratio("train", y_train)
class_ratio("test",  y_test)

## 2. Why accuracy is a trap

Consider the laziest possible "model": **always predict the majority class (0 / not fraud).**

Because ~95% of the test set really is class 0, this scores **~95% accuracy** — which sounds excellent — while catching **zero** fraud. Its recall on the positive class is **0**.

The right metrics under imbalance focus on the rare positive:

$$\text{Precision} = \frac{TP}{TP+FP}, \qquad \text{Recall} = \frac{TP}{TP+FN}, \qquad F_1 = 2\cdot\frac{P\cdot R}{P+R}$$

Precision asks *"when we shout fraud, are we right?"*; recall asks *"of all real fraud, how much did we catch?"*. The majority baseline has no positive predictions at all, so both collapse.

In [ ]:
# The trivial baseline: predict 0 for every test row (no fraud, ever).
y_pred_majority = np.zeros_like(y_test)   # all zeros, same shape as y_test

acc_maj  = accuracy_score(y_test, y_pred_majority)
# zero_division=0 keeps precision/recall/F1 at 0 (instead of warning) when there are no positive predictions.
prec_maj = precision_score(y_test, y_pred_majority, zero_division=0)
rec_maj  = recall_score(y_test, y_pred_majority, zero_division=0)
f1_maj   = f1_score(y_test, y_pred_majority, zero_division=0)

print("Trivial 'always predict majority (not fraud)' baseline:")
print(f"  accuracy : {acc_maj:.3f}   <-- looks great, but...")
print(f"  precision: {prec_maj:.3f}")
print(f"  recall   : {rec_maj:.3f}   <-- catches 0% of fraud")
print(f"  F1       : {f1_maj:.3f}")
print("\nHigh accuracy is entirely from the 95% majority. The model is useless for fraud.")

### A plain classifier is better — but still biased toward the majority

Now train an ordinary `LogisticRegression` with the default 0.5 threshold and **no** imbalance handling. It will beat the trivial baseline, but because the loss is dominated by the 95% negatives, it stays conservative about calling fraud — recall on the positive class is still low.

We keep this as our **`baseline` (untreated)** model to compare the three remedies against.

In [ ]:
# Plain logistic regression, no reweighting, no resampling. max_iter bumped so it converges cleanly.
clf_base = LogisticRegression(max_iter=2000, random_state=SEED)
clf_base.fit(X_train, y_train)                         # trained on the raw imbalanced train set

# predict_proba[:, 1] = probability of the POSITIVE (fraud) class -> needed for PR curves & thresholding.
proba_base = clf_base.predict_proba(X_test)[:, 1]
pred_base  = (proba_base >= 0.5).astype(int)           # default 0.5 threshold

print("Plain LogisticRegression (default 0.5 threshold, no imbalance handling):")
print(f"  accuracy : {accuracy_score(y_test, pred_base):.3f}")
print(f"  precision: {precision_score(y_test, pred_base, zero_division=0):.3f}")
print(f"  recall   : {recall_score(y_test, pred_base, zero_division=0):.3f}   <-- still misses lots of fraud")
print(f"  F1       : {f1_score(y_test, pred_base, zero_division=0):.3f}")
print(f"  avg prec : {average_precision_score(y_test, proba_base):.3f}   (area under PR curve)")

### Why precision-recall, not ROC or accuracy?

- **Accuracy** is dominated by the majority class, so it barely moves whatever we do to the minority.
- **ROC / AUC** plots the true-positive rate against the *false-positive rate*. The false-positive rate has the huge negative count in its denominator, so even thousands of false alarms barely nudge it — ROC looks optimistic under heavy imbalance.
- **Precision-recall** curves plot precision against recall, both of which are computed **only from quantities involving the rare positive** ($TP$, $FP$, $FN$). They react sharply to how the model treats the minority, which is exactly what we care about.

The single-number summary of a PR curve is **average precision** = $\sum_n (R_n - R_{n-1})\,P_n$, the area under the PR curve. That's our headline metric.

## 3. Remedy 1 — `class_weight='balanced'`

Instead of changing the *data*, we change the *loss*: penalize mistakes on the rare class more heavily. With `class_weight='balanced'`, scikit-learn sets each class weight inversely proportional to its frequency:

$$w_c = \frac{n_{\text{samples}}}{n_{\text{classes}} \cdot n_c}$$

so a single fraud example counts ~19x as much as a single legit example (since fraud is ~1/20 as common). The model is now pushed to catch fraud, **raising recall** — usually at the cost of some precision (more false alarms).

**Trade-off:** cheap (one argument, no extra data), no synthetic points, but it bluntly reweights *every* minority example equally and can over-predict the positive class.

In [ ]:
# Same model, one extra argument: reweight the loss by inverse class frequency.
clf_weight = LogisticRegression(max_iter=2000, random_state=SEED, class_weight="balanced")
clf_weight.fit(X_train, y_train)                       # NOTE: trained on the ORIGINAL train set (no resampling)

proba_weight = clf_weight.predict_proba(X_test)[:, 1]
pred_weight  = (proba_weight >= 0.5).astype(int)

print("class_weight='balanced' (default 0.5 threshold):")
print(f"  precision: {precision_score(y_test, pred_weight, zero_division=0):.3f}")
print(f"  recall   : {recall_score(y_test, pred_weight, zero_division=0):.3f}   <-- recall jumps up")
print(f"  F1       : {f1_score(y_test, pred_weight, zero_division=0):.3f}")
print(f"  avg prec : {average_precision_score(y_test, proba_weight):.3f}")

## 4. Remedy 2 — SMOTE (Synthetic Minority Over-sampling)

SMOTE rebalances the **training data** by *inventing* new minority examples instead of copying existing ones. For each minority sample it:

1. finds its $k$ nearest **minority** neighbours,
2. picks one neighbour at random,
3. creates a synthetic point somewhere on the line segment between them:

$$x_{\text{new}} = x_i + \lambda \,(x_{\text{neighbour}} - x_i), \qquad \lambda \sim \mathcal{U}(0,1)$$

This fills in the minority region with plausible variations rather than exact duplicates (which would just cause overfitting).

**Critical:** SMOTE is applied to **X_train / y_train only**. Synthesizing test points would leak information and inflate the scores — the test set must stay the real 95/5.

Below we try `imblearn.over_sampling.SMOTE`; if it isn't installed we fall back to a compact hand-rolled version with identical logic.

In [ ]:
# --- Try the real library first; fall back to a hand-rolled SMOTE if imblearn is missing. ---
try:
    from imblearn.over_sampling import SMOTE            # the standard implementation
    _HAVE_IMBLEARN = True

    def smote_resample(X, y, k=5, seed=SEED):
        """Balance the minority class up to the majority count using imblearn's SMOTE."""
        sm = SMOTE(random_state=seed, k_neighbors=k)
        return sm.fit_resample(X, y)

except ImportError:
    _HAVE_IMBLEARN = False

    def smote_resample(X, y, k=5, seed=SEED):
        """Hand-rolled SMOTE: oversample the minority class up to the majority count.

        For each synthetic point we pick a random minority sample, one of its k nearest
        minority neighbours, and interpolate a random fraction of the way along that segment.
        """
        gen = np.random.default_rng(seed)               # local RNG for reproducibility
        X = np.asarray(X); y = np.asarray(y)

        # Identify the minority vs majority class by counting labels.
        classes, counts = np.unique(y, return_counts=True)
        minority_class = classes[np.argmin(counts)]     # label with the fewest rows
        n_min, n_maj  = counts.min(), counts.max()
        n_to_make = n_maj - n_min                       # how many synthetic points to reach balance

        X_min = X[y == minority_class]                  # just the minority feature rows

        # Fit nearest-neighbours WITHIN the minority set. n_neighbors=k+1 because the first
        # neighbour returned for a point is the point itself (distance 0) -> we drop it.
        nn = NearestNeighbors(n_neighbors=k + 1).fit(X_min)
        neigh_idx = nn.kneighbors(X_min, return_distance=False)[:, 1:]   # (n_min, k), self dropped

        # Choose, for each of the n_to_make synthetic points, a base minority sample at random.
        base_rows = gen.integers(0, n_min, size=n_to_make)              # index into X_min
        # For each base sample, pick one of its k neighbours at random.
        pick      = gen.integers(0, k, size=n_to_make)                  # which neighbour column
        neigh_rows = neigh_idx[base_rows, pick]                         # index into X_min for the neighbour

        # Interpolate a random fraction lambda in [0,1) along base -> neighbour segment.
        lam = gen.random((n_to_make, 1))                                # (n_to_make, 1) broadcasts over features
        X_synth = X_min[base_rows] + lam * (X_min[neigh_rows] - X_min[base_rows])

        # Stack originals + synthetic minority points; label the synthetic ones as minority.
        X_res = np.vstack([X, X_synth])
        y_res = np.concatenate([y, np.full(n_to_make, minority_class)])
        return X_res, y_res

print(f"Using {'imblearn SMOTE' if _HAVE_IMBLEARN else 'HAND-ROLLED SMOTE fallback'}.")

In [ ]:
# Resample the TRAIN set only. The test set is never touched.
X_train_sm, y_train_sm = smote_resample(X_train, y_train, k=5, seed=SEED)

print("Train set BEFORE SMOTE:")
class_ratio("train", y_train)
print("Train set AFTER SMOTE (minority synthesized up to majority count):")
class_ratio("train", y_train_sm)

# Train a fresh classifier on the balanced (resampled) data. No class_weight here -
# the data itself is now balanced, so we let the default loss do its job.
clf_smote = LogisticRegression(max_iter=2000, random_state=SEED)
clf_smote.fit(X_train_sm, y_train_sm)

# ...but evaluate on the ORIGINAL, still-imbalanced test set.
proba_smote = clf_smote.predict_proba(X_test)[:, 1]
pred_smote  = (proba_smote >= 0.5).astype(int)

print("\nSMOTE-trained model (default 0.5 threshold, evaluated on real 95/5 test set):")
print(f"  precision: {precision_score(y_test, pred_smote, zero_division=0):.3f}")
print(f"  recall   : {recall_score(y_test, pred_smote, zero_division=0):.3f}")
print(f"  F1       : {f1_score(y_test, pred_smote, zero_division=0):.3f}")
print(f"  avg prec : {average_precision_score(y_test, proba_smote):.3f}")

## 5. Remedy 3 — Threshold tuning

The first two remedies changed *how the model learns*. Threshold tuning changes *how we act on its scores* — and it's free: no retraining.

By default we label positive when $p \ge 0.5$. But 0.5 is arbitrary. The `precision_recall_curve` gives precision and recall at **every** possible threshold, so we can pick the operating point that fits the business need. A common automatic choice is the threshold that **maximizes $F_1$** (best precision/recall balance).

We tune the threshold on the plain **baseline** model's scores to isolate the effect: same model, better decision rule.

In [ ]:
# precision_recall_curve returns arrays of precision/recall for a series of thresholds.
# Note: it returns len(thresholds)+1 precision/recall values (the last point is recall=0),
# so we align by slicing precision[:-1], recall[:-1] against thresholds.
prec_arr, rec_arr, thresholds = precision_recall_curve(y_test, proba_base)

# Compute F1 at every threshold (guard the 0/0 case where precision+recall == 0).
p, r = prec_arr[:-1], rec_arr[:-1]
f1_arr = np.where((p + r) > 0, 2 * p * r / (p + r), 0.0)

best_i = int(np.argmax(f1_arr))          # index of the best-F1 threshold
best_threshold = thresholds[best_i]      # <-- our tuned decision threshold (off 0.5)

# Apply the tuned threshold to the SAME baseline probabilities.
pred_thresh = (proba_base >= best_threshold).astype(int)

print(f"Tuned threshold (max F1) = {best_threshold:.3f}  (vs default 0.500)")
print("Baseline model, TUNED threshold:")
print(f"  precision: {precision_score(y_test, pred_thresh, zero_division=0):.3f}")
print(f"  recall   : {recall_score(y_test, pred_thresh, zero_division=0):.3f}")
print(f"  F1       : {f1_score(y_test, pred_thresh, zero_division=0):.3f}   <-- up from the 0.5 baseline")
print(f"  avg prec : {average_precision_score(y_test, proba_base):.3f}   (AP is threshold-free, so unchanged)")

## 6. Evaluate with precision-recall curves

A confusion matrix / F1 is a single *point*; a PR curve shows the **whole trade-off** available from a model's scores. We overlay the PR curves for all four score-producing models on one axes. The **average precision (AP)** in the legend is the area under each curve — higher and further to the top-right is better.

(Threshold tuning reuses the baseline's scores, so its PR *curve* is identical to the baseline; tuning only picks a different point *on* that curve. We therefore plot baseline, class-weight and SMOTE, and mark the tuned operating point.)

In [ ]:
plt.figure(figsize=(8, 6))

# Helper: draw one model's PR curve with its average precision in the label.
def plot_pr(proba, label, **kw):
    pr, rc, _ = precision_recall_curve(y_test, proba)   # precision & recall across thresholds
    ap = average_precision_score(y_test, proba)         # area under this PR curve
    plt.plot(rc, pr, label=f"{label} (AP={ap:.3f})", **kw)
    return ap

plot_pr(proba_base,   "baseline (untreated)", color="#888888", linestyle="--")
plot_pr(proba_weight, "class_weight='balanced'", color="#1f77b4")
plot_pr(proba_smote,  "SMOTE",                 color="#2ca02c")

# Mark the tuned operating point on the baseline curve (recall vs precision at best_threshold).
plt.scatter(recall_score(y_test, pred_thresh),
            precision_score(y_test, pred_thresh, zero_division=0),
            color="red", zorder=5, s=80, marker="*",
            label=f"tuned threshold={best_threshold:.2f} (on baseline)")

# A no-skill classifier scores precision = positive prevalence at every recall (a flat line).
prevalence = y_test.mean()
plt.axhline(prevalence, color="black", linestyle=":", linewidth=1,
            label=f"no-skill (prevalence={prevalence:.3f})")

plt.xlabel("Recall (fraction of fraud caught)")
plt.ylabel("Precision (fraction of fraud alerts that are right)")
plt.title("Precision-Recall curves under 95/5 imbalance")
plt.legend(loc="upper right")
plt.tight_layout()
plt.show()

## 7. Confusion matrices at the chosen thresholds

PR curves summarize all thresholds; a confusion matrix shows the concrete decisions at **one** operating point. Compare the untreated baseline (0.5) against each remedy. Watch the **bottom-left cell (false negatives = missed fraud)** shrink as we handle imbalance — that's the whole point.

In [ ]:
# Four panels: baseline@0.5, class_weight@0.5, SMOTE@0.5, baseline@tuned-threshold.
panels = [
    ("baseline @ 0.5",            pred_base),
    ("class_weight @ 0.5",        pred_weight),
    ("SMOTE @ 0.5",               pred_smote),
    (f"baseline @ {best_threshold:.2f}", pred_thresh),
]

fig, axes = plt.subplots(1, 4, figsize=(18, 4))
for ax, (title, preds) in zip(axes, panels):
    cm = confusion_matrix(y_test, preds)               # rows = actual, cols = predicted
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", cbar=False, ax=ax,
                xticklabels=["legit", "fraud"], yticklabels=["legit", "fraud"])
    ax.set_title(title)
    ax.set_xlabel("predicted")
    ax.set_ylabel("actual")
plt.tight_layout()
plt.show()

## 8. Side-by-side scoreboard

Finally, one table. Note how **accuracy barely distinguishes the models** (the trap!), while **recall, F1 and average precision** clearly separate the untreated baseline from the remedies.

In [ ]:
def scores(y_true, preds, proba):
    """Bundle the five metrics for one model into a dict."""
    return {
        "accuracy":  accuracy_score(y_true, preds),
        "precision": precision_score(y_true, preds, zero_division=0),
        "recall":    recall_score(y_true, preds, zero_division=0),
        "F1":        f1_score(y_true, preds, zero_division=0),
        "avg_prec":  average_precision_score(y_true, proba),
    }

results = pd.DataFrame({
    "majority-baseline":        scores(y_test, y_pred_majority, np.zeros_like(y_test, dtype=float)),
    "logreg (untreated)":       scores(y_test, pred_base,   proba_base),
    "class_weight='balanced'":  scores(y_test, pred_weight, proba_weight),
    "SMOTE":                    scores(y_test, pred_smote,  proba_smote),
    "threshold-tuned":          scores(y_test, pred_thresh, proba_base),
}).T.round(3)   # transpose so each row is a model

print(results.to_string())

# Concrete improvement of the best remedy over the untreated baseline, on our headline metrics.
base_row = results.loc["logreg (untreated)"]
print("\nImprovement over untreated baseline:")
for name in ["class_weight='balanced'", "SMOTE", "threshold-tuned"]:
    row = results.loc[name]
    print(f"  {name:24s}  recall {base_row['recall']:.3f} -> {row['recall']:.3f}   "
          f"F1 {base_row['F1']:.3f} -> {row['F1']:.3f}   "
          f"AP {base_row['avg_prec']:.3f} -> {row['avg_prec']:.3f}")

## Takeaways

- **Accuracy lies under imbalance.** A do-nothing majority predictor scored ~95% accuracy with 0% recall. Always report precision, recall, F1 and **average precision** for the rare class.
- **Precision-recall > ROC/accuracy** when positives are rare, because both axes are computed from quantities that involve the minority class, so the curve reacts to how the model treats it.
- **The three remedies attack the problem at different stages:**
  - `class_weight='balanced'` — reweights the **loss**; one line, no synthetic data, but a blunt instrument.
  - **SMOTE** — rebalances the **training data** with interpolated synthetic minority points; richer than duplication, but can synthesize into noisy regions. **Train-set only.**
  - **Threshold tuning** — changes the **decision rule** on existing scores; free, no retraining, and lets you dial the precision/recall trade-off to the business cost of a miss vs. a false alarm.
- **Golden rule:** resample / rebalance the **training set only**; keep the test set at the real prevalence or your metrics are fiction.
- These remedies are **complementary** — in practice you often combine class weights or SMOTE with a tuned threshold.